# NAVROS en Kaggle

1. **Settings → Accelerator → GPU** (T4 x2 o P100) y **Internet: On**.
2. Si el repositorio es privado, crea un secreto `GITHUB_TOKEN` en **Add-ons → Secrets**.
3. Ejecuta todo. La sesión de Kaggle dura como máximo ~12 h; NAVROS guarda checkpoint en cada ronda.
4. Para **continuar en otra sesión**: guarda la versión del notebook (*Save Version*), luego añade su
   *Output* como dataset de entrada (**Add Input**). La celda de reanudación lo detecta y sigue desde ahí.

In [ ]:
import os, shutil, subprocess
REPO = "vareoz/EMERGENCY"
BRANCH = "main"          # o la rama donde esté NAVROS
PRESET = "small"         # T4/P100: small. Con más tiempo/VRAM: base
RUN = "/kaggle/working/runs/navros"
HOURS = 11               # deja margen antes del límite de 12 h

token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
auth = f"{token}@" if token else ""
if not os.path.exists("/kaggle/working/EMERGENCY"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, f"https://{auth}github.com/{REPO}.git",
                    "/kaggle/working/EMERGENCY"], check=True)
os.chdir("/kaggle/working/EMERGENCY")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Reanudar desde la salida de una sesión anterior (si se añadió como input)
import glob
prev = [p for p in glob.glob("/kaggle/input/**/runs/navros/model.pt", recursive=True)]
if prev and not os.path.exists(f"{RUN}/model.pt"):
    shutil.copytree(os.path.dirname(prev[0]), RUN)
    print("Reanudando desde", os.path.dirname(prev[0]))

In [ ]:
if not os.path.exists(f"{RUN}/model.pt"):
    !python -m navros init --preset {PRESET} --run {RUN}

In [ ]:
# Automejora continua hasta agotar el presupuesto de horas
!NAVROS_COMPILE=1 python -m navros improve --run {RUN} --rounds 0 --hours {HOURS}

In [ ]:
!python -m navros status --run {RUN}
!python -m navros solve --run {RUN} 98765+4321